# AtmoSync: Micro-Climate Arbitrage Analytics
## Phase 2 - Task 1: Exploratory Data Analysis & Agro-Climatic Profiling
**Organization:** Infotact Solutions | **Role:** Data Associate L1 Intern  
**Mentor Review Track:** Statistical Validation & Spatial Arbitrage Formulation

---
### Phase 2 Objectives
1. **Agro-Climatic Profiling**: Inspect distributions of critical weather variables (temperature, relative humidity, precipitation, UV index).
2. **Seasonal Disparity**: Analyze seasonal micro-climate shifts across Indian crop cycles (Kharif, Rabi, Zaid).
3. **Crop Suitability Index (CSI)**: Compute and benchmark CSI across 6 major Indian crops (Rice, Wheat, Cotton, Maize, Groundnut, Sugarcane).
4. **Spatial Arbitrage Detection**: Identify nearby agricultural hub pairs with significant growing-condition differentials within 500 km.
5. **Hypothesis Testing**: Formally test regional variance via one-way ANOVA and pairwise differences via Welch's t-test.

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

# Add src to search path
sys.path.append(os.path.abspath("../src"))
from data_loader import load_dataset
from arbitrage import (
    CROPS,
    COLS,
    calculate_crop_suitability,
    haversine_distance,
    find_microclimate_arbitrage,
    run_regional_anova,
    run_location_ttest,
)

print("Phase 2 Analytics Environment Initialized.")

### 1. Load Dataset
Load data using `load_dataset()`. Automatically handles raw Kaggle repository data or synthetic Indian multi-location time-series.

In [ ]:
df, is_synthetic = load_dataset()
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns. Synthetic: {is_synthetic}")

# Ensure date and season columns are parsed
df["last_updated"] = pd.to_datetime(df["last_updated"], errors="coerce")
df["month"] = df["last_updated"].dt.month

season_map = {m: "Kharif (Jun-Oct)" for m in range(6, 11)}
season_map.update({m: "Rabi (Nov-Mar)" for m in (11, 12, 1, 2, 3)})
season_map.update({4: "Zaid (Apr-May)", 5: "Zaid (Apr-May)"})
df["season"] = df["month"].map(season_map)
df.head()

### 2. Meteorological Feature Distribution Summary

In [ ]:
weather_cols = list(COLS.values())
print("Descriptive Statistics for Meteorological Indicators:")
display(df[weather_cols].describe().round(2))

### 3. Compute Crop Suitability Index (CSI) for All Crops
Evaluate CSI across all configured crops and observe mean suitability scores.

In [ ]:
csi_results = {}
for crop_name in CROPS:
    df[f"csi_{crop_name.lower()}"] = calculate_crop_suitability(df, crop_name)
    csi_results[crop_name] = df[f"csi_{crop_name.lower()}"].mean()

csi_summary = pd.DataFrame(list(csi_results.items()), columns=["Crop", "Mean CSI (0-100)"]).sort_values(by="Mean CSI (0-100)", ascending=False)
print("Mean Crop Suitability Index Rankings:")
display(csi_summary)

### 4. Regional ANOVA Hypothesis Testing
**H0:** Regional micro-climates do not exhibit statistically significant differences in crop suitability.  
**H1:** Regional micro-climates exhibit statistically significant differences in crop suitability.

In [ ]:
df["csi"] = df["csi_rice"]
if "region" in df.columns and df["region"].nunique() > 1:
    anova_res = run_regional_anova(df, group_col="region")
    print("Regional ANOVA Results (Rice CSI):")
    print(f"  F-statistic: {anova_res['f_stat']}")
    print(f"  p-value:     {anova_res['p_val']}")
    print(f"  Significant: {anova_res['significant']} (p < 0.05)")
else:
    print("Single region in dataset; regional ANOVA skipped.")

### 5. Spatial Micro-Climate Arbitrage Opportunity Identification
Aggregate by location and find nearby location pairs with positive CSI gaps.

In [ ]:
if "location_name" in df.columns and df["location_name"].nunique() > 1:
    agg_cols = {
        "csi": "mean",
        COLS["temp"]: "mean",
        COLS["hum"]: "mean",
        COLS["rain"]: "mean",
        COLS["uv"]: "mean",
        "latitude": "first",
        "longitude": "first",
    }
    if "region" in df.columns:
        agg_cols["region"] = "first"

    loc_df = df.groupby("location_name", as_index=False).agg(agg_cols)
    loc_df = loc_df.rename(columns={
        COLS["temp"]: "temp",
        COLS["hum"]: "hum",
        COLS["rain"]: "rain",
        COLS["uv"]: "uv",
    })

    focal_loc = loc_df["location_name"].iloc[0]
    origin, arbitrage_opps = find_microclimate_arbitrage(loc_df, origin_location=focal_loc, max_radius_km=600.0)
    print(f"Arbitrage Opportunities relative to '{focal_loc}' (CSI: {origin['csi']:.1f}):")
    display(arbitrage_opps[["location_name", "distance_km", "csi", "csi_gain", "temp_diff_c", "humidity_diff"]])
else:
    print("Multi-location dataset needed for spatial arbitrage matrix.")